# RQ6 Water Intake and Obesity Levels

**Research question:** How does water consumption relate to obesity levels?

This Kaggle notebook loads the raw dataset, generates the publication-ready table as CSV, saves the figure as PDF, and prints the evidence-based answer.

In [ ]:

# ============================================================
# Kaggle-ready setup: load raw obesity dataset and shared helpers
# ============================================================
import os, glob, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from scipy.stats import chi2_contingency, pearsonr, spearmanr, f_oneway

# Kaggle output directory
OUTPUT_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('outputs')
TABLE_DIR = OUTPUT_DIR / 'tables'
FIGURE_DIR = OUTPUT_DIR / 'figures'
TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

# Change this only if needed. On Kaggle, the dataset usually lives under /kaggle/input/...
DATASET_PATH = None


def find_dataset_file():
    """Find the obesity dataset on Kaggle or locally. Supports CSV and Excel."""
    if DATASET_PATH:
        return DATASET_PATH
    patterns = [
        '/kaggle/input/**/*.csv', '/kaggle/input/**/*.xlsx', '/kaggle/input/**/*.xls',
        './*.csv', './*.xlsx', './*.xls',
        '../input/**/*.csv', '../input/**/*.xlsx', '../input/**/*.xls'
    ]
    candidates = []
    for pattern in patterns:
        candidates.extend(glob.glob(pattern, recursive=True))
    # Prefer files whose name looks like the obesity dataset
    ranked = sorted(candidates, key=lambda p: ('obesity' not in os.path.basename(p).lower(), len(p)))
    if not ranked:
        raise FileNotFoundError('No CSV/XLSX dataset file found. Upload the raw dataset to Kaggle Input or set DATASET_PATH manually.')
    return ranked[0]


def load_dataset():
    path = find_dataset_file()
    print(f'Loading dataset from: {path}')
    if path.lower().endswith(('.xlsx', '.xls')):
        df = pd.read_excel(path)
    else:
        df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]
    print(f'Dataset shape: {df.shape[0]} rows × {df.shape[1]} columns')
    return df


def clean_dataset(df):
    """Basic cleaning only; avoids changing the research meaning of variables."""
    df = df.copy()
    df = df.drop_duplicates().reset_index(drop=True)
    # Normalize object columns
    for col in df.select_dtypes(include='object').columns:
        df[col] = df[col].astype(str).str.strip()
    return df


def add_obesity_group(df):
    """Collapse detailed target classes into paper-friendly groups."""
    df = df.copy()
    mapping = {
        'Insufficient_Weight': 'Underweight',
        'Normal_Weight': 'Normal',
        'Overweight_Level_I': 'Overweight',
        'Overweight_Level_II': 'Overweight',
        'Obesity_Type_I': 'Obese',
        'Obesity_Type_II': 'Obese',
        'Obesity_Type_III': 'Obese'
    }
    df['Obesity_Group'] = df['NObeyesdad'].map(mapping).fillna(df['NObeyesdad'])
    order = ['Underweight', 'Normal', 'Overweight', 'Obese']
    df['Obesity_Group'] = pd.Categorical(df['Obesity_Group'], categories=order, ordered=True)
    return df


def add_obesity_score(df):
    """Ordinal score for correlation/ANOVA summaries."""
    df = df.copy()
    score_map = {
        'Insufficient_Weight': 0,
        'Normal_Weight': 1,
        'Overweight_Level_I': 2,
        'Overweight_Level_II': 3,
        'Obesity_Type_I': 4,
        'Obesity_Type_II': 5,
        'Obesity_Type_III': 6
    }
    df['Obesity_Score'] = df['NObeyesdad'].map(score_map)
    return df


def save_table(df, filename):
    path = TABLE_DIR / filename
    df.to_csv(path, index=False)
    print(f'Saved table: {path}')
    return path


def save_figure(filename):
    path = FIGURE_DIR / filename
    plt.tight_layout()
    plt.savefig(path, format='pdf', bbox_inches='tight')
    print(f'Saved figure: {path}')
    plt.show()
    return path


def percent_yes(series):
    return (series.astype(str).str.lower().eq('yes').mean() * 100)


def style_plot(title, xlabel=None, ylabel=None):
    plt.title(title, fontsize=14, weight='bold')
    if xlabel: plt.xlabel(xlabel, fontsize=11)
    if ylabel: plt.ylabel(ylabel, fontsize=11)
    plt.xticks(fontsize=10)
    plt.yticks(fontsize=10)
    plt.grid(axis='y', alpha=0.25)
    for spine in ['top', 'right']:
        plt.gca().spines[spine].set_visible(False)


df = add_obesity_score(add_obesity_group(clean_dataset(load_dataset())))
display(df.head())
print(df['NObeyesdad'].value_counts())


## Analysis, table, figure, and answer

In [ ]:

# RQ6: Water consumption and obesity levels
# Variables: CH2O, NObeyesdad

df_rq = df.copy()
df_rq['Water_Intake_Level'] = pd.cut(df_rq['CH2O'], bins=[0, 1.5, 2.5, np.inf], labels=['Low', 'Medium', 'High'], include_lowest=True)

ct = pd.crosstab(df_rq['Water_Intake_Level'], df_rq['Obesity_Group'])
pct = ct.div(ct.sum(axis=1), axis=0).mul(100).reset_index().round(2)
pct['sample_size'] = ct.sum(axis=1).values
avg = df_rq.groupby('Water_Intake_Level', observed=False).agg(avg_CH2O=('CH2O', 'mean'), avg_obesity_score=('Obesity_Score', 'mean')).reset_index().round(3)
summary = pct.merge(avg, on='Water_Intake_Level')
save_table(summary, 'RQ6_water_intake_obesity_distribution.csv')
display(summary)

# ANOVA across water intake groups
samples = [g['Obesity_Score'].dropna().values for _, g in df_rq.groupby('Water_Intake_Level', observed=False)]
anova = f_oneway(*samples)
test_table = pd.DataFrame({'test':['ANOVA: obesity score by water intake level'], 'f_statistic':[anova.statistic], 'p_value':[anova.pvalue]}).round(5)
save_table(test_table, 'RQ6_water_intake_anova.csv')
display(test_table)

# Figure: line chart of obese percentage by water intake
fig_df = summary.copy()
plt.figure(figsize=(8.5, 5.5))
plt.plot(fig_df['Water_Intake_Level'].astype(str), fig_df['Obese'], marker='o', linewidth=2)
style_plot('Obesity Prevalence by Water Intake Level', 'Water intake level', 'Obese (%)')
save_figure('RQ6_water_intake_obesity_line.pdf')

print('\nANSWER TO RQ6')
min_row = fig_df.loc[fig_df['Obese'].idxmin()]
max_row = fig_df.loc[fig_df['Obese'].idxmax()]
print(f"Lowest obese percentage: {min_row['Water_Intake_Level']} water intake ({min_row['Obese']:.2f}%).")
print(f"Highest obese percentage: {max_row['Water_Intake_Level']} water intake ({max_row['Obese']:.2f}%).")
print(f"ANOVA p-value: {anova.pvalue:.4g}.")
print('The table and line chart show whether obesity prevalence changes across water-consumption levels.')
